# Cookie Cats A/B Test

## Gate Placement and Player Retention

**Notebook:** 03_business_analysis.ipynb  
**Phase:** 2 — Business Analysis  
**Source:** BigQuery table `cookie_cats.clean_cookie_cats`, 90,189 players, built from Kaggle "Mobile Games: A/B Testing" (`cookie_cats.csv`), originally from DataCamp project 184  
**Author:** Taofeek Salami  
**Date:** 2026-10-20  

## Describe the Test Groups and Their Outcomes

This notebook first checks that the group sizes match the planned 50/50 split (sample ratio test), then describes 1-day retention, 7-day retention and game rounds by version. All results are descriptive: they record what each version showed and do not test whether the differences are statistically significant. The queries are the same as in `sql/02_business_analysis.sql`.

In [1]:
import os                            # reads settings from environment variables
from pathlib import Path             # builds file paths from the project root

from dotenv import load_dotenv       # loads the .env file into environment variables
from google.cloud import bigquery    # Google's Python client for BigQuery
from scipy.stats import chisquare    # chi-square goodness-of-fit test for the sample ratio check

## Configuration

Settings come from `.env`. The sample ratio settings are the values fixed in `PROJECT_INFO.md` → Decision Rules before the test was run.

In [2]:
# The notebook runs from notebooks/, so the project root is one folder up.
PROJECT_ROOT = Path.cwd().resolve().parent
load_dotenv(PROJECT_ROOT / ".env")

PROJECT_ID = os.environ["GCP_PROJECT_ID"]
DATASET = os.environ["BQ_DATASET"]
LOCATION = os.environ["BQ_LOCATION"]

TABLE_ID = f"{PROJECT_ID}.{DATASET}.clean_cookie_cats"

# Sample ratio test settings (PROJECT_INFO.md → Decision Rules).
EXPECTED_SHARE = {"gate_30": 0.5, "gate_40": 0.5}   # assumed 50/50 allocation (A-009)
SRM_ALERT_P = 0.001                                  # below this, comparisons between versions are paused

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
print(TABLE_ID)

cookie-cats-ab-taofeek.cookie_cats.clean_cookie_cats


## 2.0 Sample Ratio Test

Checks whether the observed group sizes are consistent with a 50/50 split. A p-value below 0.001 would indicate a sample ratio mismatch: a problem with assignment or data collection that would make comparisons between versions unreliable. A higher p-value means no evidence of a mismatch, not proof that assignment was random.

In [3]:
# 2.0 Same count as sql/02_business_analysis.sql, Section 0.
group_sizes = client.query(f"""
    SELECT version, COUNT(*) AS players
    FROM `{TABLE_ID}`
    GROUP BY version
    ORDER BY version
""").to_dataframe()

total_players = group_sizes["players"].sum()
group_sizes["share"] = group_sizes["players"] / total_players
group_sizes["expected_players"] = group_sizes["version"].map(EXPECTED_SHARE) * total_players

# chisquare compares observed with expected counts. With two groups the test has
# one degree of freedom. Expected counts must add up to the observed total,
# which they do because the expected shares sum to 1.
statistic, p_value = chisquare(f_obs=group_sizes["players"], f_exp=group_sizes["expected_players"])

print(group_sizes.to_string(index=False))
print(f"\nChi-square statistic: {statistic:.3f}")
print(f"p-value: {p_value:.4f}")
if p_value < SRM_ALERT_P:
    print("Sample ratio mismatch: comparisons between versions are paused")
else:
    print(f"No evidence of a sample ratio mismatch at the {SRM_ALERT_P} threshold")

version  players     share  expected_players
gate_30    44700  0.495626           45094.5
gate_40    45489  0.504374           45094.5

Chi-square statistic: 6.902
p-value: 0.0086
No evidence of a sample ratio mismatch at the 0.001 threshold


**Result.** gate_30 has 44,700 players (49.56%) and gate_40 has 45,489 (50.44%), against 45,094.5 each under a 50/50 split. The chi-square statistic is 6.902 with p = 0.0086. This is above the 0.001 alert threshold set in Decision Rules, so no sample ratio mismatch is flagged and the version comparisons go ahead. The p-value is below 0.05, so the imbalance is unusual for a true 50/50 split. It is reported here and carried into the Phase 4 validity notes. Passing this check does not show that assignment was random (A-002).